# Feature Extraction for Fake News Detection

This notebook demonstrates feature extraction techniques:
1. **TF-IDF** - Term Frequency-Inverse Document Frequency (converts text to numerical features)
2. **Word Embeddings** - Word2Vec distributed representations
3. **BERT Embeddings** - Contextual embeddings from pre-trained BERT

Using **real data** from the combined dataset (44,898 articles)

## Part 1: Setup and Imports

In [4]:
import feature_extraction
import bert_utils
import preprocessing
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import time

warnings.filterwarnings('ignore')

## Part 2: Load Dataset

In [6]:
preprocessed_path = 'data/preprocessed_data.csv'

df = pd.read_csv(preprocessed_path)

print(f"\nUsing ENTIRE preprocessed dataset ({len(df):,} articles)...")
texts = df['content'].tolist()
labels = df['label'].tolist()

print(f"\n--- Sample REAL article (PREPROCESSED) ---")
real_idx = df[df['label'] == 1].index[0]
print(df.loc[real_idx, 'content'][:300])

print(f"\n--- Sample FAKE article (PREPROCESSED) ---")
fake_idx = df[df['label'] == 0].index[0]
print(df.loc[fake_idx, 'content'][:300])


Using ENTIRE preprocessed dataset (44,889 articles)...

--- Sample REAL article (PREPROCESSED) ---
us budget fight loom republican flip fiscal script washington reuter head conserv republican faction us congress vote month huge expans nation debt pay tax cut call fiscal conserv sunday urg budget restraint keep sharp pivot way among republican us repres mark meadow speak cb face nation drew hard l

--- Sample FAKE article (PREPROCESSED) ---
donald trump send embarrass new year eve messag disturb donald trump wish american happi new year leav instead give shout enemi hater dishonest fake news media former realiti show star one job countri rapidli grow stronger smarter want wish friend support enemi hater even dishonest fake news media h


## Part 3: TF-IDF Feature Extraction

In [8]:
tfidf = feature_extraction.TFIDFExtractor(max_features=5000)
tfidf_features = tfidf.fit_transform(texts)

print(f"\nTop 20 Terms from Vocabulary:")
feature_names = tfidf.get_feature_names()
for i, name in enumerate(feature_names[:20]):
    print(f"  {i+1:2d}. {name}")

INFO:feature_extraction:Initialized TF-IDF extractor (max_features=5000)
INFO:feature_extraction:Fitting and transforming 44889 documents:
INFO:feature_extraction:Features shape: (44889, 5000)



Top 20 Terms from Vocabulary:
   1. abandon
   2. abba
   3. abc
   4. abc news
   5. abdullah
   6. abe
   7. abedin
   8. abid
   9. abil
  10. abl
  11. abort
  12. abroad
  13. absenc
  14. absolut
  15. absurd
  16. abu
  17. abus
  18. academ
  19. academi
  20. acceler


## Part 4: BERT Embeddings Feature Extraction

In [12]:

texts_demo = texts[:10]

bert = feature_extraction.BERTEmbeddings()
bert_features = bert.extract_embeddings_batch(texts_demo, batch_size=5)


print(f"\nEmbedding Matrix:")
print(f"  Shape: {bert_features.shape}")
print(f"  Rows (documents): {bert_features.shape[0]}")
print(f"  Columns (embedding dimensions): {bert_features.shape[1]}")
print(f"  Data type: {bert_features.dtype}")
print(f"\nEmbedding Statistics:")
print(f"  Mean: {bert_features.mean():.4f}")
print(f"  Std: {bert_features.std():.4f}")
print(f"  Min: {bert_features.min():.4f}")
print(f"  Max: {bert_features.max():.4f}")

INFO:feature_extraction:Initialized BERT embeddings (model=bert-base-uncased, device=cpu)



Embedding Matrix:
  Shape: (10, 768)
  Rows (documents): 10
  Columns (embedding dimensions): 768
  Data type: float32

Embedding Statistics:
  Mean: -0.0486
  Std: 0.8444
  Min: -26.3003
  Max: 1.5644


## Part 5: Feature Statistics

In [16]:

print("SAVING FEATURE EXTRACTION RESULTS (FULL DATASET)")

Path('features').mkdir(exist_ok=True)

# Save TF-IDF features from FULL dataset
tfidf_save_path = 'features/tfidf_features_full.npy'
np.save(tfidf_save_path, tfidf_features)
print(f"\n Saved FULL TF-IDF features ({tfidf_features.shape}):")
print(f"  Path: {tfidf_save_path}")
print(f"  Size: {Path(tfidf_save_path).stat().st_size / (1024*1024):.2f} MB")
print(f"  Articles: {tfidf_features.shape[0]:,}")
print(f"  Features: {tfidf_features.shape[1]:,}")

# Save TF-IDF feature names (vocabulary)
tfidf_vocab_path = 'features/tfidf_vocabulary.txt'
with open(tfidf_vocab_path, 'w') as f:
    for name in feature_names:
        f.write(name + '\n')
print(f"\n Saved TF-IDF vocabulary ({len(feature_names):,} terms):")
print(f"  Path: {tfidf_vocab_path}")

# Save labels (full dataset)
labels_path = 'features/labels.npy'
np.save(labels_path, np.array(labels))
print(f"\n Saved labels for FULL dataset ({len(labels):,} articles):")
print(f"  Path: {labels_path}")
print(f"  Real (1): {sum(1 for l in labels if l == 1):,}")
print(f"  Fake (0): {sum(1 for l in labels if l == 0):,}")

# Save extraction metadata (convert numpy types to native Python types for JSON)
import json
metadata = {
    'dataset': {
        'total_articles': int(len(df)),
        'real_articles': int((df['label'] == 1).sum()),
        'fake_articles': int((df['label'] == 0).sum())
    },
    'tfidf': {
        'shape': list(tfidf_features.shape),
        'max_features': 5000,
        'ngram_range': [1, 2],
        'sparsity_pct': float(100 * (tfidf_features == 0).sum() / tfidf_features.size)
    },
    'bert': {
        'model': 'bert-base-uncased',
        'embedding_dim': 768,
        'pooling': 'mean'
    }
}

metadata_path = 'features/extraction_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"\n Saved extraction metadata:")
print(f"  Path: {metadata_path}")


SAVING FEATURE EXTRACTION RESULTS (FULL DATASET)

 Saved FULL TF-IDF features ((44889, 5000)):
  Path: features/tfidf_features_full.npy
  Size: 1712.38 MB
  Articles: 44,889
  Features: 5,000

 Saved TF-IDF vocabulary (5,000 terms):
  Path: features/tfidf_vocabulary.txt

 Saved labels for FULL dataset (44,889 articles):
  Path: features/labels.npy
  Real (1): 21,417
  Fake (0): 23,472

 Saved extraction metadata:
  Path: features/extraction_metadata.json
